In [1]:
import cv2
import mediapipe as mp
import numpy as np
import json
import time
import os
from collections import defaultdict


In [2]:
mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils
pose = mp_pose.Pose()


In [3]:
def calculate_angle(a, b, c):
    a, b, c = map(np.array, (a, b, c))
    ba = a - b
    bc = c - b
    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))
    return angle


In [4]:
def get_angle_value(lm, angle_name):
    P = mp_pose.PoseLandmark

    mapping = {
        "left_elbow": (P.LEFT_SHOULDER, P.LEFT_ELBOW, P.LEFT_WRIST),
        "right_elbow": (P.RIGHT_SHOULDER, P.RIGHT_ELBOW, P.RIGHT_WRIST),
        "left_knee": (P.LEFT_HIP, P.LEFT_KNEE, P.LEFT_ANKLE),
        "right_knee": (P.RIGHT_HIP, P.RIGHT_KNEE, P.RIGHT_ANKLE),
        "left_shoulder": (P.LEFT_ELBOW, P.LEFT_SHOULDER, P.LEFT_HIP),
        "right_shoulder": (P.RIGHT_ELBOW, P.RIGHT_SHOULDER, P.RIGHT_HIP),
        "left_hip": (P.LEFT_SHOULDER, P.LEFT_HIP, P.LEFT_KNEE),
        "right_hip": (P.RIGHT_SHOULDER, P.RIGHT_HIP, P.RIGHT_KNEE),
    }

    if angle_name not in mapping:
        return None

    a, b, c = mapping[angle_name]
    return calculate_angle(
        [lm[a].x, lm[a].y],
        [lm[b].x, lm[b].y],
        [lm[c].x, lm[c].y]
    )


In [5]:
''' SESSION_TIME = 60

asana_config = [
    ("POSE1", "pose1_pranamasana.json"),
    ("POSE2", "pose2_values.json"),
    ("POSE3", "pose3_padahastasana.json"),
    ("POSE4", "pose4_ashwa_sanchalanasana.json"),
]

joint_highlight_map = {
    "left_elbow": mp_pose.PoseLandmark.LEFT_ELBOW,
    "right_elbow": mp_pose.PoseLandmark.RIGHT_ELBOW,
    "left_knee": mp_pose.PoseLandmark.LEFT_KNEE,
    "right_knee": mp_pose.PoseLandmark.RIGHT_KNEE,
    "left_shoulder": mp_pose.PoseLandmark.LEFT_SHOULDER,
    "right_shoulder": mp_pose.PoseLandmark.RIGHT_SHOULDER,
    "left_hip": mp_pose.PoseLandmark.LEFT_HIP,
    "right_hip": mp_pose.PoseLandmark.RIGHT_HIP,
}

cap = cv2.VideoCapture(0)

final_results = {}

for folder_name, json_file in asana_config:

    with open(json_file, "r") as f:
        pose_data = json.load(f)

    pose_name = pose_data["pose_name"]
    pose_ranges = pose_data["angles"]

    print(f"\nStarting {pose_name}")

    image_files = os.listdir(folder_name)
    reference_path = os.path.join(folder_name, image_files[0])
    reference_img = cv2.imread(reference_path)
    reference_img = cv2.resize(reference_img, (640,480))

    ref_rgb = cv2.cvtColor(reference_img, cv2.COLOR_BGR2RGB)
    ref_results = pose.process(ref_rgb)
    ideal_skeleton = reference_img.copy()

    if ref_results.pose_landmarks:
        mp_draw.draw_landmarks(
            ideal_skeleton,
            ref_results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )

    session_start = time.time()

    total_frames = 0
    correct_frames = 0
    angle_history = defaultdict(list)

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        total_frames += 1

        frame = cv2.resize(frame, (640,480))
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)

        user_skeleton = frame.copy()
        pose_correct = True

        if results.pose_landmarks:
            lm = results.pose_landmarks.landmark

            mp_draw.draw_landmarks(
                user_skeleton,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS
            )

            for angle_name, (min_v, max_v) in pose_ranges.items():

                angle_val = get_angle_value(lm, angle_name)

                if angle_val is None:
                    continue

                angle_history[angle_name].append(angle_val)

                if not (min_v <= angle_val <= max_v):
                    pose_correct = False

                    if angle_name in joint_highlight_map:
                        idx = joint_highlight_map[angle_name]
                        h, w, _ = frame.shape
                        x = int(lm[idx].x * w)
                        y = int(lm[idx].y * h)
                        cv2.circle(user_skeleton, (x,y), 15, (0,0,255), -1)

        if pose_correct:
            correct_frames += 1

        elapsed = int(time.time() - session_start)

        top_row = cv2.hconcat([frame, reference_img])
        bottom_row = cv2.hconcat([ideal_skeleton, user_skeleton])
        final_display = cv2.vconcat([top_row, bottom_row])

        cv2.putText(final_display, pose_name, (20,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,0), 2)

        cv2.imshow("AI Yoga Trainer", final_display)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            exit()

        if elapsed >= SESSION_TIME:
            break

    # ---------- CALCULATE METRICS ----------
    accuracy = round((correct_frames / total_frames) * 100, 2)

    variations = []
    for angle_vals in angle_history.values():
        if len(angle_vals) > 1:
            variations.append(np.std(angle_vals))

    stability = round(max(0, 100 - np.mean(variations)*5), 2) if variations else 100

    final_results[pose_name] = {
        "accuracy": accuracy,
        "stability": stability
    }

# ---------- FINAL REPORT ----------
print("\n===== FINAL YOGA REPORT =====")
for pose_name, metrics in final_results.items():
    print(f"{pose_name}")
    print(f"Accuracy: {metrics['accuracy']}%")
    print(f"Stability: {metrics['stability']}%")
    print("---------------------------")

cap.release()
cv2.destroyAllWindows()'''


' SESSION_TIME = 60\n\nasana_config = [\n    ("POSE1", "pose1_pranamasana.json"),\n    ("POSE2", "pose2_values.json"),\n    ("POSE3", "pose3_padahastasana.json"),\n    ("POSE4", "pose4_ashwa_sanchalanasana.json"),\n]\n\njoint_highlight_map = {\n    "left_elbow": mp_pose.PoseLandmark.LEFT_ELBOW,\n    "right_elbow": mp_pose.PoseLandmark.RIGHT_ELBOW,\n    "left_knee": mp_pose.PoseLandmark.LEFT_KNEE,\n    "right_knee": mp_pose.PoseLandmark.RIGHT_KNEE,\n    "left_shoulder": mp_pose.PoseLandmark.LEFT_SHOULDER,\n    "right_shoulder": mp_pose.PoseLandmark.RIGHT_SHOULDER,\n    "left_hip": mp_pose.PoseLandmark.LEFT_HIP,\n    "right_hip": mp_pose.PoseLandmark.RIGHT_HIP,\n}\n\ncap = cv2.VideoCapture(0)\n\nfinal_results = {}\n\nfor folder_name, json_file in asana_config:\n\n    with open(json_file, "r") as f:\n        pose_data = json.load(f)\n\n    pose_name = pose_data["pose_name"]\n    pose_ranges = pose_data["angles"]\n\n    print(f"\nStarting {pose_name}")\n\n    image_files = os.listdir(folde

In [ ]:
SESSION_TIME = 500

asana_config = [
    ("POSE1", "pose1_pranamasana.json"),
    ("POSE2", "pose2_ values.json"),
    ("POSE3", "pose3_padahastasana.json"),
    ("POSE4", "pose4_ashwa_sanchalanasana.json")
]

joint_highlight_map = {
    "left_elbow": mp_pose.PoseLandmark.LEFT_ELBOW,
    "right_elbow": mp_pose.PoseLandmark.RIGHT_ELBOW,
    "left_knee": mp_pose.PoseLandmark.LEFT_KNEE,
    "right_knee": mp_pose.PoseLandmark.RIGHT_KNEE,
    "left_shoulder": mp_pose.PoseLandmark.LEFT_SHOULDER,
    "right_shoulder": mp_pose.PoseLandmark.RIGHT_SHOULDER,
    "left_hip": mp_pose.PoseLandmark.LEFT_HIP,
    "right_hip": mp_pose.PoseLandmark.RIGHT_HIP,
}

cap = cv2.VideoCapture(0)
final_report = {}

for folder_name, json_file in asana_config:

    with open(json_file, "r") as f:
        pose_data = json.load(f)

    pose_name = pose_data["pose_name"]
    pose_ranges = pose_data["angles"]

    print(f"\n🧘 Starting {pose_name}")

    # -------- Load Sample Image --------
    image_files = os.listdir(folder_name)
    reference_path = os.path.join(folder_name, image_files[0])

    sample_img = cv2.imread(reference_path)
    sample_img = cv2.resize(sample_img, (640,360))

    sample_rgb = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
    sample_results = pose.process(sample_rgb)

    sample_skeleton = sample_img.copy()

    if sample_results.pose_landmarks:
        mp_draw.draw_landmarks(
            sample_skeleton,
            sample_results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )

    session_start = time.time()
    pose_hold_start = None
    correct_hold_time = 0

    total_frames = 0
    correct_frames = 0
    angle_history = defaultdict(list)

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (640,480))
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)

        error_panel = frame.copy()
        pose_correct = False   # default False

        current_time = time.time()
        elapsed = int(current_time - session_start)

        feedback = []

        # ---------- Person Detected ----------
        if results.pose_landmarks:

            total_frames += 1
            pose_correct = True

            lm = results.pose_landmarks.landmark

            for angle_name, (min_v, max_v) in pose_ranges.items():

                angle_val = get_angle_value(lm, angle_name)

                if angle_val is None:
                    pose_correct = False
                    continue

                angle_history[angle_name].append(angle_val)

                if not (min_v <= angle_val <= max_v):
                    pose_correct = False
                    feedback.append(f"Adjust {angle_name.replace('_',' ')}")

                    if angle_name in joint_highlight_map:
                        idx = joint_highlight_map[angle_name]
                        h, w, _ = frame.shape
                        x = int(lm[idx].x * w)
                        y = int(lm[idx].y * h)
                        cv2.circle(error_panel, (x,y), 20, (0,0,255), -1)

            # ---------- Hold Timer ----------
            if pose_correct:
                correct_frames += 1
                if pose_hold_start is None:
                    pose_hold_start = current_time
            else:
                if pose_hold_start:
                    correct_hold_time += current_time - pose_hold_start
                    pose_hold_start = None

        else:
            feedback.append("Person not detected")
            if pose_hold_start:
                correct_hold_time += current_time - pose_hold_start
                pose_hold_start = None

        # Current hold display
        if pose_hold_start:
            current_hold = int(current_time - pose_hold_start)
        else:
            current_hold = 0

        # ---------- Display ----------
        cv2.putText(frame, pose_name, (20,35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,0), 2)

        cv2.putText(frame, f"Session: {elapsed}/{SESSION_TIME}s",
                    (20,75), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        cv2.putText(frame, f"Correct Hold: {int(correct_hold_time + current_hold)}s",
                    (20,105), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

        y = 140
        for msg in feedback:
            cv2.putText(frame, msg, (20,y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
            y += 25

        # ---------- Layout ----------
        live_small = cv2.resize(frame, (640,360))
        ideal_small = sample_skeleton
        error_big = cv2.resize(error_panel, (1280,480))

        top_row = cv2.hconcat([live_small, ideal_small])
        final_display = cv2.vconcat([top_row, error_big])

        cv2.imshow("AI Yoga Trainer", final_display)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            exit()

        if elapsed >= SESSION_TIME:
            break

    # ---------- Metrics ----------
    if total_frames > 0:
        accuracy = round((correct_frames / total_frames) * 100, 2)
    else:
        accuracy = 0

    variations = []
    for vals in angle_history.values():
        if len(vals) > 1:
            variations.append(np.std(vals))

    stability = round(max(0, 100 - np.mean(variations)*5), 2) if variations else 0

    final_report[pose_name] = {
        "accuracy": accuracy,
        "stability": stability,
        "correct_hold_time": round(correct_hold_time,2)
    }

    # ---------- Relax ----------
    relax_start = time.time()
    while time.time() - relax_start < RELAX_TIME:
        ret, frame = cap.read()
        if not ret:
            break

        remaining = RELAX_TIME - int(time.time() - relax_start)

        cv2.putText(frame, "RELAX", (200,200),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (0,255,255), 3)
        cv2.putText(frame, f"Next pose in {remaining}s", (200,260),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

        cv2.imshow("AI Yoga Trainer", frame)
        cv2.waitKey(1)

# ---------- Final Report ----------
print("\n===== FINAL YOGA PERFORMANCE REPORT =====\n")

for pose_name, metrics in final_report.items():
    print(f"{pose_name}")
    print(f"  Accuracy        : {metrics['accuracy']}%")
    print(f"  Stability Score : {metrics['stability']}%")
    print(f"  Correct Hold    : {metrics['correct_hold_time']} sec")
    print("------------------------------------------------")

cap.release()
cv2.destroyAllWindows()



🧘 Starting Pranamasana


NameError: name 'RELAX_TIME' is not defined

In [ ]:
SESSION_TIME = 500

asana_config = [
    ("POSE1", "pose1_pranamasana.json"),

]

joint_highlight_map = {
    "left_elbow": mp_pose.PoseLandmark.LEFT_ELBOW,
    "right_elbow": mp_pose.PoseLandmark.RIGHT_ELBOW,
    "left_knee": mp_pose.PoseLandmark.LEFT_KNEE,
    "right_knee": mp_pose.PoseLandmark.RIGHT_KNEE,
    "left_shoulder": mp_pose.PoseLandmark.LEFT_SHOULDER,
    "right_shoulder": mp_pose.PoseLandmark.RIGHT_SHOULDER,
    "left_hip": mp_pose.PoseLandmark.LEFT_HIP,
    "right_hip": mp_pose.PoseLandmark.RIGHT_HIP,
}

cap = cv2.VideoCapture(0)
final_report = {}

for folder_name, json_file in asana_config:

    with open(json_file, "r") as f:
        pose_data = json.load(f)

    pose_name = pose_data["pose_name"]
    pose_ranges = pose_data["angles"]

    print(f"\n🧘 Starting {pose_name}")

    # -------- Load Sample Image --------
    image_files = os.listdir(folder_name)
    reference_path = os.path.join(folder_name, image_files[0])

    sample_img = cv2.imread(reference_path)
    sample_img = cv2.resize(sample_img, (640,360))

    sample_rgb = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
    sample_results = pose.process(sample_rgb)

    sample_skeleton = sample_img.copy()

    if sample_results.pose_landmarks:
        mp_draw.draw_landmarks(
            sample_skeleton,
            sample_results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS
        )

    session_start = time.time()
    pose_hold_start = None
    correct_hold_time = 0

    total_frames = 0
    correct_frames = 0
    angle_history = defaultdict(list)

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (640,480))
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)

        error_panel = frame.copy()
        pose_correct = False   # default False

        current_time = time.time()
        elapsed = int(current_time - session_start)

        feedback = []

        # ---------- Person Detected ----------
        if results.pose_landmarks:

            total_frames += 1
            pose_correct = True

            lm = results.pose_landmarks.landmark

            for angle_name, (min_v, max_v) in pose_ranges.items():

                angle_val = get_angle_value(lm, angle_name)

                if angle_val is None:
                    pose_correct = False
                    continue

                angle_history[angle_name].append(angle_val)

                if not (min_v <= angle_val <= max_v):
                    pose_correct = False
                    feedback.append(f"Adjust {angle_name.replace('_',' ')}")

                    if angle_name in joint_highlight_map:
                        idx = joint_highlight_map[angle_name]
                        h, w, _ = frame.shape
                        x = int(lm[idx].x * w)
                        y = int(lm[idx].y * h)
                        cv2.circle(error_panel, (x,y), 20, (0,0,255), -1)

            # ---------- Hold Timer ----------
            if pose_correct:
                correct_frames += 1
                if pose_hold_start is None:
                    pose_hold_start = current_time
            else:
                if pose_hold_start:
                    correct_hold_time += current_time - pose_hold_start
                    pose_hold_start = None

        else:
            feedback.append("Person not detected")
            if pose_hold_start:
                correct_hold_time += current_time - pose_hold_start
                pose_hold_start = None

        # Current hold display
        if pose_hold_start:
            current_hold = int(current_time - pose_hold_start)
        else:
            current_hold = 0

        # ---------- Display ----------
        cv2.putText(frame, pose_name, (20,35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,0), 2)

        cv2.putText(frame, f"Session: {elapsed}/{SESSION_TIME}s",
                    (20,75), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        cv2.putText(frame, f"Correct Hold: {int(correct_hold_time + current_hold)}s",
                    (20,105), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

        y = 140
        for msg in feedback:
            cv2.putText(frame, msg, (20,y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
            y += 25

        # ---------- Layout ----------
        live_small = cv2.resize(frame, (640,360))
        ideal_small = sample_skeleton
        error_big = cv2.resize(error_panel, (1280,480))

        top_row = cv2.hconcat([live_small, ideal_small])
        final_display = cv2.vconcat([top_row, error_big])

        cv2.imshow("AI Yoga Trainer", final_display)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            exit()

        if elapsed >= SESSION_TIME:
            break

    # ---------- Metrics ----------
    if total_frames > 0:
        accuracy = round((correct_frames / total_frames) * 100, 2)
    else:
        accuracy = 0

    variations = []
    for vals in angle_history.values():
        if len(vals) > 1:
            variations.append(np.std(vals))

    stability = round(max(0, 100 - np.mean(variations)*5), 2) if variations else 0

    final_report[pose_name] = {
        "accuracy": accuracy,
        "stability": stability,
        "correct_hold_time": round(correct_hold_time,2)
    }

    # ---------- Relax ----------
    relax_start = time.time()
    while time.time() - relax_start < RELAX_TIME:
        ret, frame = cap.read()
        if not ret:
            break

        remaining = RELAX_TIME - int(time.time() - relax_start)

        cv2.putText(frame, "RELAX", (200,200),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (0,255,255), 3)
        cv2.putText(frame, f"Next pose in {remaining}s", (200,260),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

        cv2.imshow("AI Yoga Trainer", frame)
        cv2.waitKey(1)

# ---------- Final Report ----------
print("\n===== FINAL YOGA PERFORMANCE REPORT =====\n")

for pose_name, metrics in final_report.items():
    print(f"{pose_name}")
    print(f"  Accuracy        : {metrics['accuracy']}%")
    print(f"  Stability Score : {metrics['stability']}%")
    print(f"  Correct Hold    : {metrics['correct_hold_time']} sec")
    print("------------------------------------------------")

cap.release()
cv2.destroyAllWindows()


In [ ]:
''' SESSION_TIME = 180      # 3 minutes per pose
RELAX_TIME = 30         # 30 sec rest

asana_config = [
    ("POSE1", "pose1_pranamasana.json"),
    ("POSE2", "pose2_values.json"),
    ("POSE3", "pose3_padahastasana.json"),
    ("POSE4", "pose4_ashwa_sanchalanasana.json"),
]

joint_highlight_map = {
    "left_elbow": mp_pose.PoseLandmark.LEFT_ELBOW,
    "right_elbow": mp_pose.PoseLandmark.RIGHT_ELBOW,
    "left_knee": mp_pose.PoseLandmark.LEFT_KNEE,
    "right_knee": mp_pose.PoseLandmark.RIGHT_KNEE,
    "left_shoulder": mp_pose.PoseLandmark.LEFT_SHOULDER,
    "right_shoulder": mp_pose.PoseLandmark.RIGHT_SHOULDER,
    "left_hip": mp_pose.PoseLandmark.LEFT_HIP,
    "right_hip": mp_pose.PoseLandmark.RIGHT_HIP,
}

cap = cv2.VideoCapture(0)

for folder_name, json_file in asana_config:

    with open(json_file, "r") as f:
        pose_data = json.load(f)

    pose_name = pose_data["pose_name"]
    pose_ranges = pose_data["angles"]

    print(f"\n🧘 Starting {pose_name}")

    session_start = time.time()
    pose_hold_start = None
    correct_hold_time = 0

    while cap.isOpened():

        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (640,480))
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(frame_rgb)

        error_panel = frame.copy()
        ideal_panel = frame.copy()

        pose_correct = True
        feedback = []

        if results.pose_landmarks:
            lm = results.pose_landmarks.landmark

            mp_draw.draw_landmarks(
                ideal_panel,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS
            )

            for angle_name, (min_v, max_v) in pose_ranges.items():

                angle_val = get_angle_value(lm, angle_name)
                if angle_val is None:
                    continue

                if not (min_v <= angle_val <= max_v):
                    pose_correct = False
                    feedback.append(f"Adjust {angle_name.replace('_',' ')}")

                    if angle_name in joint_highlight_map:
                        idx = joint_highlight_map[angle_name]
                        h, w, _ = frame.shape
                        x = int(lm[idx].x * w)
                        y = int(lm[idx].y * h)
                        cv2.circle(error_panel, (x,y), 20, (0,0,255), -1)

        # -------- HOLD TIMER --------
        current_time = time.time()

        if pose_correct:
            if pose_hold_start is None:
                pose_hold_start = current_time
        else:
            if pose_hold_start:
                correct_hold_time += current_time - pose_hold_start
                pose_hold_start = None

        if pose_hold_start:
            current_hold = int(current_time - pose_hold_start)
        else:
            current_hold = 0

        elapsed = int(current_time - session_start)

        # -------- LIVE TEXT --------
        status_text = "POSE CORRECT" if pose_correct else "ADJUST POSTURE"
        status_color = (0,255,0) if pose_correct else (0,0,255)

        cv2.putText(frame, pose_name, (20,35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,0), 2)

        cv2.putText(frame, status_text, (20,75),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, status_color, 2)

        cv2.putText(frame, f"Session: {elapsed}/{SESSION_TIME}s",
                    (20,110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        cv2.putText(frame, f"Correct Hold: {int(correct_hold_time + current_hold)}s",
                    (20,140), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)

        y = 180
        for msg in feedback:
            cv2.putText(frame, msg, (20,y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
            y += 30

        # -------- RESIZE PANELS --------
        live_small = cv2.resize(frame, (640,360))
        ideal_small = cv2.resize(ideal_panel, (640,360))
        error_big = cv2.resize(error_panel, (1280,480))

        top_row = cv2.hconcat([live_small, ideal_small])
        final_display = cv2.vconcat([top_row, error_big])

        cv2.imshow("AI Yoga Trainer", final_display)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            cap.release()
            cv2.destroyAllWindows()
            exit()

        if elapsed >= SESSION_TIME:
            break

    # -------- RELAX TIME --------
    relax_start = time.time()
    while time.time() - relax_start < RELAX_TIME:
        ret, frame = cap.read()
        if not ret:
            break

        remaining = RELAX_TIME - int(time.time() - relax_start)

        cv2.putText(frame, "RELAX", (200,200),
                    cv2.FONT_HERSHEY_SIMPLEX, 2, (0,255,255), 3)
        cv2.putText(frame, f"Next pose in {remaining}s", (200,260),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

        cv2.imshow("AI Yoga Trainer", frame)
        cv2.waitKey(1)

cap.release()
cv2.destroyAllWindows()'''


' SESSION_TIME = 180      # 3 minutes per pose\nRELAX_TIME = 30         # 30 sec rest\n\nasana_config = [\n    ("POSE1", "pose1_pranamasana.json"),\n    ("POSE2", "pose2_values.json"),\n    ("POSE3", "pose3_padahastasana.json"),\n    ("POSE4", "pose4_ashwa_sanchalanasana.json"),\n]\n\njoint_highlight_map = {\n    "left_elbow": mp_pose.PoseLandmark.LEFT_ELBOW,\n    "right_elbow": mp_pose.PoseLandmark.RIGHT_ELBOW,\n    "left_knee": mp_pose.PoseLandmark.LEFT_KNEE,\n    "right_knee": mp_pose.PoseLandmark.RIGHT_KNEE,\n    "left_shoulder": mp_pose.PoseLandmark.LEFT_SHOULDER,\n    "right_shoulder": mp_pose.PoseLandmark.RIGHT_SHOULDER,\n    "left_hip": mp_pose.PoseLandmark.LEFT_HIP,\n    "right_hip": mp_pose.PoseLandmark.RIGHT_HIP,\n}\n\ncap = cv2.VideoCapture(0)\n\nfor folder_name, json_file in asana_config:\n\n    with open(json_file, "r") as f:\n        pose_data = json.load(f)\n\n    pose_name = pose_data["pose_name"]\n    pose_ranges = pose_data["angles"]\n\n    print(f"\n🧘 Starting {pose